# Benchmark Performance

Computes per-similarity-bin Pearson r and MAE for one or more models against the NTAB test set.

**Inputs** (configure in the next cell):
- `ACTIVITIES_PATH`: path to `activities.parquet` produced by `nfab_preprocess.run_pipeline`
- `MODELS`: dict mapping a display name to a predictions CSV path

Predictions CSVs must have columns: `assay_id`, `ligand_name`, `standard_type`, `pred_pchembl`.
Activity rows without a matching prediction are filled with `pred_pchembl = 6.0` (1 µM).

In [ ]:
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ntab_evaluate.plotting import _parse_split_label, load_predictions, compute_model_stats

ACTIVITIES_PATH = pathlib.Path("out/activities.parquet")

# Map model display name → path to predictions CSV.
# The CSV must have columns: assay_id, ligand_name, standard_type, pred_pchembl.
MODELS: dict[str, str | pathlib.Path] = {
    "Ligand-only FP baseline": "out_baseline/predictions_v0.csv",
    # "Ligand-only FP+Mol prop baseline": "out_baseline/predictions_v1.csv",
    "Ligand-only Mol prop baseline": "out_baseline/predictions_v2.csv",
}

MIN_ASSAY_SIZE = 10  # minimum compounds per (assay, standard_type) to include in Pearson r
N_BOOTSTRAP = 1000   # bootstrap resamples for standard error

## Load activities

In [ ]:
activities = pd.read_parquet(
    ACTIVITIES_PATH,
    columns=["assay_chembl_id", "ligand_chembl_id", "standard_type", "split", "pchembl_value_filled"],
)

test_activities = activities[activities["split"].str.startswith("test_sim_", na=False)].copy()
print(f"Total test rows: {len(test_activities):,}")
print("Splits found:")
for s, n in sorted(test_activities["split"].value_counts().items()):
    print(f"  {s}: {n:,}")

## Discover bins from split labels

In [ ]:
unique_splits = sorted(
    test_activities["split"].unique(),
    key=lambda s: _parse_split_label(s)[0],
)
bins = [_parse_split_label(s) for s in unique_splits]  # list of (lo, hi, label)

print("Bins (in plot order):")
for lo, hi, label in bins:
    n = (test_activities["split"] == (f"test_sim_{lo:.2f}" if lo == hi else f"test_sim_{lo:.2f}_{hi:.2f}")).sum()
    print(f"  {label}: {n:,} rows")

## Load predictions and compute per-bin Pearson r

In [ ]:
# tol:vibrant qualitative palette and marker shapes — same order as Runs-n-Poses
_TOL_VIBRANT = [
    "#0077BB",  # blue
    "#009988",  # teal
    "#EE7733",  # orange
    "#CC3311",  # red
    "#AA3377",  # magenta
    "#33BBEE",  # cyan
    "#BBBBBB",  # grey
]
_MARKERS = ["o", "s", "D", "X", "P", "H", "v", "^", "<", ">"]

all_per_assay: dict[str, pd.DataFrame] = {}
all_aggregated: dict[str, pd.DataFrame] = {}

for model_name, pred_path in MODELS.items():
    print(f"\nLoading predictions for: {model_name}")
    merged = load_predictions(pred_path, test_activities)
    per_assay, aggregated = compute_model_stats(merged, bins, min_assay_size=MIN_ASSAY_SIZE, n_bootstrap=N_BOOTSTRAP)
    all_per_assay[model_name] = per_assay
    all_aggregated[model_name] = aggregated
    print(aggregated.to_string(index=False))

## Plot average performance per bin

In [ ]:
fig, (ax_r, ax_mae) = plt.subplots(1, 2, figsize=(16, 6))

first_agg = next(iter(all_aggregated.values()))
x = np.arange(len(first_agg))

xticklabels = [
    f"{row.display_label}\n$n_a={row.n_a}$\n$n_m={row.n_m}$"
    for row in first_agg.itertuples()
]

for i, (model_name, agg) in enumerate(all_aggregated.items()):
    color = _TOL_VIBRANT[i % len(_TOL_VIBRANT)]
    marker = _MARKERS[i % len(_MARKERS)]

    # --- Pearson r ---
    ax_r.plot(x, agg["pearson_r"].values, marker=marker, markersize=8, linewidth=2, color=color, label=model_name)
    ax_r.fill_between(x, agg["pearson_r_ci_low"].values, agg["pearson_r_ci_high"].values, alpha=0.15, color=color)

    # --- MAE ---
    ax_mae.plot(x, agg["mae"].values, marker=marker, markersize=8, linewidth=2, color=color, label=model_name)
    ax_mae.fill_between(x, agg["mae_ci_low"].values, agg["mae_ci_high"].values, alpha=0.15, color=color)

ax_r.set_ylabel("Mean Pearson Correlation", fontsize=12, fontweight="bold")
ax_r.set_ylim(0, 1)
ax_r.tick_params(axis="y", labelsize=12)
ax_r.legend(frameon=False, fontsize=12, loc="upper left")
ax_r.spines[["top", "right"]].set_visible(False)
ax_r.grid(True, linestyle="--", alpha=0.5)
ax_r.set_xticks(x)
ax_r.set_xticklabels(xticklabels, fontsize=12)
ax_r.set_xlabel("Compound similarity to the training and val sets", fontsize=12, fontweight="bold")

ax_mae.set_ylabel("Mean Absolute Error", fontsize=12, fontweight="bold")
ax_mae.tick_params(axis="y", labelsize=12)
ax_mae.spines[["top", "right"]].set_visible(False)
ax_mae.grid(True, linestyle="--", alpha=0.5)
ax_mae.set_xticks(x)
ax_mae.set_xticklabels(xticklabels, fontsize=12)
ax_mae.set_xlabel("Compound similarity to the training and val sets", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()
fig.savefig("benchmark_performance.png", dpi=300, bbox_inches="tight")

## Per-assay distribution as box plot per bin

In [ ]:
n_models = len(all_per_assay)
n_bins = len(bins)
width = 0.8 / n_models
x = np.arange(n_bins)

first_agg = next(iter(all_aggregated.values()))
xticklabels = [
    f"{row.display_label}\n$n_a={row.n_a}$\n$n_m={row.n_m}$"
    for row in first_agg.itertuples()
]

fig, (ax_r, ax_mae) = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

for i, (model_name, per_assay) in enumerate(all_per_assay.items()):
    color = _TOL_VIBRANT[i % len(_TOL_VIBRANT)]
    positions = x + (i - (n_models - 1) / 2) * width

    box_kwargs = dict(
        positions=positions,
        widths=width * 0.85,
        patch_artist=True,
        boxprops=dict(facecolor=color, alpha=0.6),
        medianprops=dict(color="black", linewidth=1.5),
        whiskerprops=dict(color=color),
        capprops=dict(color=color),
        flierprops=dict(marker="o", markerfacecolor=color, markeredgecolor=color, markersize=3, alpha=0.5),
    )

    r_data = [
        per_assay.loc[per_assay["display_label"] == label, "pearson_r"].dropna().values
        for _, _, label in bins
    ]
    mae_data = [
        per_assay.loc[per_assay["display_label"] == label, "mae"].dropna().values
        for _, _, label in bins
    ]

    ax_r.boxplot(r_data, **box_kwargs)
    ax_mae.boxplot(mae_data, **box_kwargs)
    ax_r.plot([], [], color=color, linewidth=8, alpha=0.6, label=model_name)

ax_r.set_ylabel("Per-assay Pearson Correlation", fontsize=12, fontweight="bold")
ax_r.tick_params(axis="y", labelsize=12)
ax_r.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.4)
ax_r.legend(frameon=False, fontsize=12, loc="upper left")
ax_r.spines[["top", "right"]].set_visible(False)
ax_r.grid(True, axis="y", linestyle="--", alpha=0.5)

ax_mae.set_ylabel("Per-assay Mean Absolute Error", fontsize=12, fontweight="bold")
ax_mae.tick_params(axis="y", labelsize=12)
ax_mae.set_xlabel("Compound similarity to the training and val sets", fontsize=12, fontweight="bold")
ax_mae.set_xticks(x)
ax_mae.set_xticklabels(xticklabels, fontsize=12)
ax_mae.spines[["top", "right"]].set_visible(False)
ax_mae.grid(True, axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()
fig.savefig("benchmark_performance_boxplot.png", dpi=300, bbox_inches="tight")

## Pairwise similarity-bin comparisons with Tukey-HSD

Tests whether per-assay `TUKEY_METRIC` differs significantly between similarity bins for a single model (`TUKEY_MODEL`).

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

TUKEY_MODEL = "Ligand-only FP baseline"  # change to any key in MODELS
TUKEY_METRIC = "pearson_r"

per_assay = all_per_assay[TUKEY_MODEL].dropna(subset=[TUKEY_METRIC])
result = pairwise_tukeyhsd(
    endog=per_assay[TUKEY_METRIC].values,
    groups=per_assay["display_label"].values,
)
print(f"Model: {TUKEY_MODEL!r}\n")
print(result.summary())